**OpenMM**

In [1]:
import openmm
print(openmm.__file__)

/home/feynman/miniconda3/envs/venv/lib/python3.13/site-packages/openmm/__init__.py


In [11]:
from openmm.app import *
from openmm import *
from openmm.unit import kelvin, picosecond, nanometer
from openmm import Platform


pdb = PDBFile('/home/feynman/Downloads/input_pdb/1UBQ.pdb')

forcefield = ForceField('amber19-all.xml', 'amber19/tip3pfb.xml')

modeller = Modeller(pdb.topology, pdb.positions)
modeller.addHydrogens(forcefield)

PDBFile.writeFile(modeller.topology, modeller.positions, open('/home/feynman/Downloads/input_pdb/processed.pdb', 'w'))

system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME,
        nonbondedCutoff=1*nanometer, constraints=HBonds)

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.004*picosecond)

platform = Platform.getPlatformByName('CUDA')

simulation = Simulation(modeller.topology, system, integrator, platform)

simulation.context.setPositions(modeller.positions)

print("Minimizing...")
simulation.minimizeEnergy()

simulation.reporters.append(DCDReporter('/data/openmm_out/sample/run1/traj.dcd', 1000))

simulation.reporters.append(StateDataReporter('/data/openmm_out/sample/run1/log.txt', 1000, step=True,
        potentialEnergy=True, temperature=True))

simulation.step(100000)
print("Done")

Minimizing...
Done


In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import kelvin, picosecond, nanometer

# Load structure
pdb = PDBFile('/home/feynman/Downloads/foldx5_Linux/1PGA_Repair.pdb')

# Force field
forcefield = ForceField(
    'charmm36.xml',
    'implicit/obc2.xml'
)

# Build system
system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=NoCutoff,
    constraints=HBonds
)

# Integrator
integrator = LangevinIntegrator(
    400*kelvin,
    1/picosecond,
    0.002*picosecond
)

# Platform
platform = Platform.getPlatformByName('CUDA')

# Simulation object
simulation = Simulation(
    pdb.topology,
    system,
    integrator,
    platform
)

# Set positions
simulation.context.setPositions(pdb.positions)

# Energy minimization
print("Minimizing...")
simulation.minimizeEnergy()

# Save minimized structure
positions = simulation.context.getState(
    getPositions=True
).getPositions()

with open('minimized.pdb', 'w') as f:
    PDBFile.writeFile(
        simulation.topology,
        positions,
        f
    )

print("Minimization complete.")